In [157]:
import os
from pathlib import Path
from tqdm import tqdm
import pandas as pd

from rasterio.features import shapes
from shapely.geometry import box, shape

import sys
sys.path.append('..')  # Adjust the path as per your directory structure

from scripts.constants import *
from scripts.utils import *
from scripts.raster_operations import *
from scripts.vector_operations import *

In [149]:
# IN paths
# bua_path = BUA_IN_DIR / "BUA_2022_GB_3186007556224938771.gpkg"
bua_path = BUA_IN_DIR / "OS_Open_Built_Up_Areas.gpkg"
pch_vrt_path = PCH_IN_DIR / "ps_PSScene4Band_2019.vrt"
pch_paths = list(sorted(PCH_IN_DIR.glob('*.tif')))
gbg_5km_path = GBG_IN_DIR / "5km_grid_region.shp"
bhf_paths = list(sorted(BHF_IN_DIR.rglob('*9.gdbtable')))
ogs_path = OGS_IN_DIR / "GB_GreenspaceSite.shp"

# OUT paths
bua_dissolved_path = BUA_OUT_DIR / "bua_dissolved.geojson"
pch_masked_dir = PCH_OUT_DIR / "pch_bua_masked"
pch_masked_dir.mkdir(parents=True, exist_ok=True)
pch_masked_vrt_path = pch_masked_dir / "ps_PSScene4Band_2019_masked.vrt"
pch_masked_tile_dir = PCH_OUT_DIR / "pch_bua_masked_gbg_tile"
pch_masked_r_tile_dir = pch_masked_tile_dir / "raster_tile"
pch_masked_r_tile_dir.mkdir(parents=True, exist_ok=True)
pch_masked_v_tile_dir = pch_masked_tile_dir / "vector_tile"
pch_masked_v_tile_dir.mkdir(parents=True, exist_ok=True)
bhf_dissolved_tile_dir = BHF_OUT_DIR / "bhf_dissolved_tile"
bhf_dissolved_tile_dir.mkdir(parents=True, exist_ok=True)
bhf_distance_tile_dir = BHF_OUT_DIR / "bhf_distance_tile"
bhf_distance_tile_dir.mkdir(parents=True, exist_ok=True)

# Data Processing

## PCH and BUA Processing

### Create VRT from PCH

In [43]:
create_vrt(pch_paths, pch_vrt_path)

### Dissolve BUA

In [11]:
bua_gpd = gpd.read_file(bua_path)
bua_crs = bua_gpd.crs
bua_dissolved_gpd = bua_gpd.dissolve().reset_index(drop=True).drop(columns=['gsscode', 'name1_text', 'name1_language', 'name2_text',
       'name2_language', 'areahectares', 'geometry_area_m'])

Handle everything with pch's crs, which is EPSG:4326

In [ ]:
with rasterio.open(pch_vrt_path) as pch:
    global pch_crs
    pch_crs = pch.crs
    bua_dissolved_gpd = bua_dissolved_gpd.to_crs(pch_crs)
    bua_dissolved_gpd.to_file(bua_dissolved_path)

### Mask PCH with BUA dissolved

In [9]:
mask_path = bua_dissolved_path
min_val, max_val = 3, 70

for i, in_path in tqdm(enumerate(pch_paths)):
    
    out_name = in_path.stemb + "_masked.tif"
    out_path = pch_masked_dir / out_name
    
    clip_and_mask_raster(in_path, bua_dissolved_gpd, out_path, min_val, max_val)

82it [22:33, 16.50s/it]


#### Create VRT of PCH-masked

In [14]:
pch_masked_paths = list(sorted(pch_masked_dir.glob('*.tif')))
create_vrt(pch_masked_paths, pch_masked_vrt_path)

### Tile PCH-masked using GBG tiles (5 km)

In [59]:
gbg_5km_gdf = gpd.read_file(gbg_5km_path)
gbg_crs = gbg_5km_gdf.crs
gbg_gdf = gbg_5km_gdf.copy().to_crs(pch_crs).sort_values(by=['TILE_NAME'])

In [128]:
def vectorise_raster(raster_data, transform, field_name, crs, dissolve=True):
    # Generate vector shapes from raster
    mask = raster_data != raster_data.min()  # Mask to exclude the background/nodata values
    vector_shapes = shapes(raster_data, mask=mask, transform=transform)

    # Convert the shapes and values to a GeoDataFrame
    records = []
    for geom, value in vector_shapes:
        records.append({
            'geometry': shape(geom),
            field_name: value
        })

    # Convert the shapes and values to a GeoDataFrame
    if records:  # Check if there are any records to avoid creating an empty GeoDataFrame
        gdf = gpd.GeoDataFrame(records, crs=crs)
        gdf.set_geometry('geometry', inplace=True)  # Ensure 'geometry' is the geometry column
        gdf = gdf  # Assign CRS after confirming geometry presence

        # Optional dissolve step
        if dissolve:
            gdf = gdf.dissolve(by=field_name, as_index=False)

        return gdf
    else:
        return None  # Return None if no vector shapes were created

In [147]:
gbg_gdf[gbg_gdf['TILE_NAME'] == 'ST72NW'] # Desde ST72NW no tienen tif file
print(len(gbg_gdf))

10647


In [ ]:
# Open the raster file
error_tiles =[]
empty_tiles = []
empty_vectors = []
with rasterio.open(pch_masked_vrt_path) as src:
    gbg_gdf = gbg_gdf.to_crs(src.crs)
    for index, row in tqdm(gbg_gdf.iloc[7240:].iterrows()):
        # if index > 10:
        #     break
        tile_name = row.TILE_NAME
        output_name = f"pch_{tile_name}"
        
        # The geometry to mask the raster with

        # geom = getFeatures(gpd.GeoDataFrame([row], crs=src.crs))
        # geom = [row['geometry']] # Opcion 1 --> funciona
        temp_gdp = gpd.GeoDataFrame([row], crs=src.crs)
        geom = [feature["geometry"] for _, feature in temp_gdp.iterrows()]
        
        try:
            # print('masking...')
            out_image, out_transform = mask(src, geom, crop=True)
            if out_image.size == 0 or out_image.mean() == src.nodata:
                # print(index, "Empty raster tile", tile_name)
                empty_tiles.append(tile_name)
                continue
            # print('vectorising...')
            vector_gdf = vectorise_raster(out_image[0], out_transform, 'height', src.crs, dissolve=True)
        
            # Save vectorized output
            if vector_gdf is None:
                # print(index, "Empty vector tile", tile_name)
                empty_vectors.append(tile_name)
                continue
            print(tile_name)
            
            tile_r_code_dir = pch_masked_r_tile_dir / tile_name[:2].lower()
            tile_r_code_dir.mkdir(parents=True, exist_ok=True)
            output_r_path = tile_r_code_dir / f"{output_name}.tif"
            
            tile_v_code_dir = pch_masked_v_tile_dir / tile_name[:2].lower()
            tile_v_code_dir.mkdir(parents=True, exist_ok=True)
            output_v_path = tile_v_code_dir / f"{output_name}.geojson"
            vector_gdf.to_file(output_v_path, driver='GeoJSON')

            out_meta = src.meta.copy()

            # Update the metadata to reflect the number of layers,
            # and the new transformed affine and the new height and width
            out_meta.update({"driver": "GTiff",
                                "height": out_image.shape[1],
                                "width": out_image.shape[2],
                                "transform": out_transform})
            
            # with rasterio.open(output_r_path, "w", **out_meta) as dest:
            #     dest.write(out_image)
        
        except ValueError as e:
            print(index, e, tile_name)
            error_tiles.append(tile_name)

In [151]:
error_tiles

['HP62SW',
 'NA00SE',
 'NA10NE',
 'NA10NW',
 'NA10SE',
 'NA10SW',
 'NA64NE',
 'NA74NW',
 'NF09NE',
 'NF19NW',
 'NF56SE',
 'NF58SE',
 'NF60NW',
 'NF60SW',
 'NF66NE',
 'NF66NW',
 'NF66SE',
 'NF66SW',
 'NF67SE',
 'NF68SW',
 'NL57NE',
 'NL58NE',
 'NL58SE',
 'NL58SW',
 'NL69NW',
 'ST72SE',
 'SV80NE',
 'SV80NW']

## BHF Processing

Dissolve BHF with itself keeping non-adjacent polygons as separate features

In [154]:
for bhf_path in tqdm(bhf_paths[0:1]):
    
    bhf_tile_out_dir = bhf_dissolved_tile_dir / bhf_path.parents[1].name
    bhf_tile_out_dir.mkdir(parents=True, exist_ok=True)
    bhf_tile_out_path = bhf_tile_out_dir / f"bhf_dissolved_{bhf_path.parent.name[:6]}.geojson"
    
    bhf_gpd = gpd.read_file(bhf_path)

    bhf_dissolved_gpd = dissolve_adjacent(bhf_gpd).to_crs(pch_crs)

    bhf_dissolved_gpd.to_file(bhf_tile_out_path)

bhf_crs = bhf_gpd.crs

100%|██████████| 1/1 [00:00<00:00, 149.32it/s]


## OGS Processing

In [152]:
ogs_gdf = gpd.read_file(ogs_path)
ogs_crs = ogs_gdf.crs

In [ ]:
ogs_gdf = ogs_gdf.to_crs(pch_crs)
for i, row in tqdm(gbg_gdf.iterrows()):
    # Assuming we're using the first feature in the clipping layer
    tile_name = row.TILE_NAME
    ogs_tile_out_dir = OGS_OUT_DIR / tile_name[:2].lower()
    ogs_tile_out_dir.mkdir(parents=True, exist_ok=True)
    ogs_tile_out_path = ogs_tile_out_dir / f"ogs_clipped_{tile_name}.geojson"
    
    bounding_box = row.geometry.bounds
    # Convert the bounding box to a polygon
    clip_extent_polygon = box(*bounding_box)
    # Clip the layer
    ogs_clipped_gdf = gpd.clip(ogs_gdf, clip_extent_polygon)

    if len(ogs_clipped_gdf) > 0:

        ogs_clipped_gdf.to_file(ogs_tile_out_path)

# 3-30-300 Estimations

Create a DataFrame that gathers all the file paths generated in the processing steps to determine which tiles have the three types of data:

- PCH masked vector tiles
- BHF dissolved tiles
- OGS clipped tiles

Note that GBG has two systems for naming tiles, which are translated using the `translate_tile_name()` function from `utils.py`

In [156]:
pch_tile_paths = list(sorted(pch_masked_v_tile_dir.rglob('*.geojson')))
bhf_tile_paths = list(sorted(BHF_OUT_DIR.rglob('*.geojson')))
ogs_tile_paths = list(sorted(OGS_OUT_DIR.rglob('*.geojson')))

In [161]:
pch_tile_paths_df = pd.DataFrame([{'big_tile': p.stem[-6:-4].lower(), 'direction_code': p.stem[-6:], 'path': str(p)} for p in pch_tile_paths])
pch_tile_paths_df['number_code'] = pch_tile_paths_df.apply(lambda x: translate_tile_name(x.direction_code), axis=1)

bhf_tile_paths_df = pd.DataFrame([{'big_tile': p.stem[-6:-4], 'number_code': p.stem[-6:], 'path': str(p)} for p in bhf_tile_paths])
bhf_tile_paths_df['direction_code'] = bhf_tile_paths_df.apply(lambda x: translate_tile_name(x.number_code), axis=1)

ogs_tile_paths_df = pd.DataFrame([{'big_tile': p.parent.stem, 'direction_code': p.stem[-6:], 'path': str(p)} for p in ogs_tile_paths])
ogs_tile_paths_df['number_code'] = ogs_tile_paths_df.apply(lambda x: translate_tile_name(x.direction_code), axis=1)

In [ ]:
columns = ['big_tile', 'number_code', 'direction_code', 'path_pch', 'path_bhf', 'path_ogs']
tile_paths_df = pch_tile_paths_df.merge(
    bhf_tile_paths_df.merge(ogs_tile_paths_df, how='outer', on=['big_tile', 'direction_code', 'number_code'], suffixes=['_bhf', '_ogs']),
    how='outer', on=['big_tile', 'direction_code', 'number_code']
    ).rename(columns={'path': 'path_pch'})[columns]
tile_paths_df

In [171]:
tile_paths_filtered_df = tile_paths_df.copy()[~tile_paths_df['path_bhf'].isna()]
tile_paths_filtered_df

,big_tile,number_code,direction_code,path_pch,path_bhf,path_ogs
0,hp,hp4500,HP40SE,NaN,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,NaN
1,hp,hp5505,HP50NE,NaN,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,NaN
2,hp,hp5005,HP50NW,NaN,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,NaN
3,hp,hp5500,HP50SE,NaN,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...
4,hp,hp5500,HP50SE,NaN,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...
...,...,...,...,...,...,...
9107,tv,tv4595,TV49NE,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...
9108,tv,tv4095,TV49NW,NaN,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,NaN
9109,tv,tv5595,TV59NE,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...
9110,tv,tv5095,TV59NW,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...,/Users/ancazugo/Documents/PhD_Thesis/Tree_dete...


In [172]:
from shapely.ops import nearest_points

# Define a function that returns the closest point and distance for a given geometry
def find_closest_point_and_distance(reference_layer, compare_layer):

    single_polygon = compare_layer[compare_layer['height'] != 255].dissolve()
    geom = single_polygon.geometry.iloc[0]
    reference_point, closest_point = nearest_points(reference_layer, geom)
    distance = reference_layer.distance(closest_point)
    return pd.Series([distance], index=['distance_pch'])

In [173]:
def estimate_distance(row, crs=27700):
    bhf_tile_gdf = gpd.read_file(row.path_bhf).to_crs(crs)

    bhf_distance_gdf = gpd.GeoDataFrame()

    if row['path_pch'] is not np.nan:

        pch_tile_gdf = gpd.read_file(row.path_pch).to_crs(crs)
        closest_pch_gdf = bhf_tile_gdf.copy()
        closest_pch_gdf['distance_pch'] = closest_pch_gdf.apply(lambda row: find_closest_point_and_distance(row.geometry, pch_tile_gdf), axis=1)
        bhf_distance_gdf = closest_pch_gdf[['distance_pch', 'geometry']]

    if row['path_ogs'] is not np.nan:    

        ogs_tile_gdf = gpd.read_file(row.path_ogs).to_crs(crs)
        nearest_ogs = gpd.sjoin_nearest(bhf_tile_gdf, ogs_tile_gdf, distance_col='distance_ogs', how='left')
        nearest_ogs = nearest_ogs[['id', 'function', 'distName1', 'distName2', 'distName3', 'distName4', 'distance_ogs', 'geometry']]

        if row['path_pch'] is not np.nan:
            bhf_distance_gdf = bhf_distance_gdf.merge(nearest_ogs, on='geometry', how='outer')
        else:
            bhf_distance_gdf = nearest_ogs

    if len(bhf_distance_gdf) == 0:
        return None

    bhf_distance_subtile_dir = bhf_distance_tile_dir / row.big_tile
    bhf_distance_subtile_dir.mkdir(parents=True, exist_ok=True)
    bhf_distance_path = bhf_distance_subtile_dir / f"bhf_distance_{row.number_code}.geojson"
    bhf_distance_gdf.to_file(bhf_distance_path)

    return bhf_distance_gdf

In [175]:
from concurrent.futures import ThreadPoolExecutor
cpu_cores_os = os.cpu_count()
cpu_cores_os

16

In [ ]:
%%time
with ThreadPoolExecutor(max_workers=cpu_cores_os) as executor:
    results = list(executor.map(estimate_distance, [row for _, row in tile_paths_filtered_df.iterrows()]))

## 3-rule (Three trees in sight)

## 30-rule (30% canopy cover in neighbourhood)

## 300-rule (300 m from the closest public park)

# Notes
## Data Sources

- PCH: [Liu et al. 2023](https://zenodo.org/records/8154445) (UK - 2019)
- BUA: [Digimap](https://digimap.edina.ac.uk/) (GB - December 2022)
- GBG: [Digimap](https://digimap.edina.ac.uk/) (GB - 2012)
- BHF: [Digimap](https://digimap.edina.ac.uk/) (GB - October 2017)
- OGS: [Digimap](https://digimap.edina.ac.uk/) (GB - October 2017)
- IMD: [Consumer Data Research Center](https://data.cdrc.ac.uk/dataset/index-multiple-deprivation-imd) (UK - 2019-2020)
  - LSOA Boundaries (England and Wales) [Ministry of Housing, Communities and Local Government](https://data-communities.opendata.arcgis.com/datasets/6bced6c6f81448cf9692ed3f472b11ce/explore?location=52.661142%2C-2.243978%2C7.00) (2019)
  - Zone Boundaries (Scotland) [www.data.gov.uk](https://www.data.gov.uk/dataset/ab9f1f20-3b7f-4efa-9bd2-239acf63b540/data-zone-boundaries-2011) (2011)

## CRS
All the data processing steps were performed using a standard CRS of *EPSG:4326* (degrees) for all layers. To calculate distances, vector layers were transformed to *EPSG:27700* (metres).

In [155]:
print('PCH:', pch_crs)
print('BUA:', bua_crs)
print('GBG:', gbg_crs)
print('BHF:', bhf_crs)
print('OGS:', ogs_crs)

PCH: EPSG:4326
BUA: EPSG:27700
GBG: PROJCS["OSGB36 / British National Grid",GEOGCS["OSGB36",DATUM["Ordnance_Survey_of_Great_Britain_1936",SPHEROID["Airy 1830",6377563.396,299.3249646,AUTHORITY["EPSG","7001"]],AUTHORITY["EPSG","6277"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",49],PARAMETER["central_meridian",-2],PARAMETER["scale_factor",0.999601272],PARAMETER["false_easting",400000],PARAMETER["false_northing",-100000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
BHF: EPSG:27700
OGS: EPSG:27700
